In [ ]:
import os
repo_url = "https://github.com/aaronjji/rk8391-devbox.git"
if not os.path.exists("/kaggle/working/gave2-challenge"):
    os.system(f"git clone --recurse-submodules {repo_url} /kaggle/working/gave2-challenge")
os.chdir("/kaggle/working/gave2-challenge")
os.system("git log -1 --oneline")
print(os.getcwd())

In [ ]:
os.system("pip install -q albumentations opencv-python-headless scikit-image scipy networkx sknw huggingface_hub safetensors")

In [ ]:
candidates = [
    "/kaggle/input/datasets/aaronajit/gave2-preliminary/GAVE2_preliminary",
    "/kaggle/input/gave2-preliminary/GAVE2_preliminary",
]
os.system("find /kaggle/input -maxdepth 4")
DATA_ROOT = None
for c in candidates:
    if os.path.exists(f"{c}/training/images"):
        DATA_ROOT = c
        break
assert DATA_ROOT is not None, f"Dataset not found in {candidates}"
print("DATA_ROOT =", DATA_ROOT)

reg_candidates = [
    "/kaggle/input/datasets/aaronajit/gave2-registered/registered",
    "/kaggle/input/gave2-registered/registered",
]
FFA_ROOT = None
for c in reg_candidates:
    if os.path.exists(f"{c}/training/FFA_A"):
        FFA_ROOT = c
        break
assert FFA_ROOT is not None, f"Registered FFA dataset not found in {reg_candidates}"
print("FFA_ROOT =", FFA_ROOT)

xattn_ckpt_candidates = [
    "/kaggle/input/datasets/aaronajit/gave2-task2-xattn-fold0-ckpt/latest.pth",
    "/kaggle/input/gave2-task2-xattn-fold0-ckpt/latest.pth",
]
XATTN_CKPT = None
for c in xattn_ckpt_candidates:
    if os.path.exists(c):
        XATTN_CKPT = c
        break
assert XATTN_CKPT is not None, f"xattn fold0 checkpoint not found in {xattn_ckpt_candidates}"
print("XATTN_CKPT =", XATTN_CKPT)

# Place it where --resume expects it (out-dir/fold0/latest.pth) before training resumes
os.makedirs("runs/task2_xattn_proto/fold0", exist_ok=True)
import shutil
shutil.copy(XATTN_CKPT, "runs/task2_xattn_proto/fold0/latest.pth")
print("staged latest.pth for resume")

In [ ]:
# Continue the cross-attention fusion prototype from epoch 30 (already validated
# as a strong go-signal on COR/INF vs the additive baseline) to the full 60
# epochs, for an apples-to-apples comparison before the big ensemble session.
FOLD = 0
os.system(
    f"python -u src/train_task2.py "
    f"--fold {FOLD} --seed 77 --data-root {DATA_ROOT} --ffa-root {FFA_ROOT} "
    f"--fusion xattn "
    f"--patch-size 320 "
    f"--epochs 60 --steps-per-epoch 50 --num-workers 2 "
    f"--vein-pos-weight 6.0 --vein-topology-ratio 1.3 "
    f"--checkpoint-every-epochs 5 --val-every-epochs 5 --max-seconds 10800 "
    f"--out-dir runs/task2_xattn_proto --resume"
)